# 5章　エラー、ログ、デバッグ

### 5.1.1　Pythonのエラーメッセージ

In [1]:
# defの行に引数dataを書き忘れたもの
from scipy.stats import linregress

def fit_trendline(year_timestamps):
    result = linregress(year_timestamps, data)
    slope = round(result.slope, 3)
    r_squared = round(result.rvalue**2, 3)
    return slope, r_squared

In [2]:
women_in_parliament=[9.02, 9.01, 8.84, 8.84, 8.84]
timestamps= [2000, 2001, 2002, 2003, 2004]

In [3]:
fit_trendline(timestamps)

NameError: name 'data' is not defined

In [4]:
# fit_trendlineのdefを修正
from scipy.stats import linregress

def fit_trendline(year_timestamps, data):
    result = linregress(year_timestamps, data)
    slope = round(result.slope, 3)
    r_squared = round(result.rvalue**2, 3)
    return slope, r_squared

In [5]:
# うまく結果が出る
fit_trendline(timestamps, women_in_parliament)

(np.float64(-0.053), np.float64(0.763))

In [6]:
# timespampsの要素を文字列にして実行 -> エラーが起こる
women_in_parliament=[9.02, 9.01, 8.84, 8.84, 8.84]
timestamps= ["2000", "2001", "2002", "2003", "2004", "2005"]
fit_trendline(timestamps, women_in_parliament)

UFuncTypeError: ufunc 'maximum' did not contain a loop with signature matching types (dtype('<U4'), dtype('<U4')) -> None

UFuncTypeErrorはTypeErrorの一種なので、TypeErrorが起こったときの処理を加える

In [7]:
from scipy.stats import linregress

def fit_trendline(year_timestamps, data):
    try:
        result = linregress(year_timestamps, data)
    except TypeError:
        print("*ERROR*: Both lists must contain only float or integers")
    else:
        slope = round(result.slope, 3)
        r_squared = round(result.rvalue**2, 3)
        return slope, r_squared

In [8]:
women_in_parliament=[9.02, 9.01, 8.84, 8.84, 8.84]
timestamps= ["2000", "2001", "2002", "2003", "2004"]
fit_trendline(timestamps, women_in_parliament)

*ERROR*: Both lists must contain only float or integers


In [9]:
# エラーが起こってもデフォルト値を返す
def fit_trendline(year_timestamps, data):
    try:
        result = linregress(year_timestamps, data)
    except TypeError:
        print("*ERROR*: Both lists must contain only float or integers")
        return 0.0, 0.0
    else:
        slope = round(result.slope, 3)
        r_squared = round(result.rvalue**2, 3)
        return slope, r_squared

In [10]:
fit_trendline(timestamps, women_in_parliament)

*ERROR*: Both lists must contain only float or integers


(0.0, 0.0)

### 5.1.3　エラーの生成

In [11]:
from scipy.stats import linregress

def fit_trendline(year_timestamps, data):
    if not year_timestamps or data:
        raise ValueError("Timestamps and data cannot be empty lists")
    result = linregress(year_timestamps, data)
    slope = round(result.slope, 3)
    r_squared = round(result.rvalue**2, 3)
    return slope, r_squared

In [5]:
women_in_parliament=[9.02, 9.01, 8.84, 8.84, 8.84]
timestamps= ["2000", "2001", "2002", "2003", "2004", "2005"]
fit_trendline(timestamps, women_in_parliament)

UFuncTypeError: ufunc 'maximum' did not contain a loop with signature matching types (dtype('<U4'), dtype('<U4')) -> None

## 5.2　ロギング

### 5.2.2 ロギングの設定

In [1]:
import logging

In [2]:
logging.basicConfig(level=logging.DEBUG)

In [3]:
logging.basicConfig(filename="chapter_05_logs.log", level=logging.DEBUG)

In [5]:
from scipy.stats import linregress

def fit_trendline(year_timestamps, data):
    logging.info("Running fit_trendline function")
    result = linregress(year_timestamps, data)
    slope = round(result.slope, 3)
    r_squared = round(result.rvalue**2, 3)
    logging.info(f"Completed analysis. Slope of the trendline is {slope}.")
    return slope, r_squared

In [6]:
women_in_parliament=[9.02, 9.01, 8.84, 8.84, 8.84]
timestamps= [2000, 2001, 2002, 2003, 2004]
fit_trendline(timestamps, women_in_parliament)

INFO:root:Running fit_trendline function
INFO:root:Completed analysis. Slope of the trendline is -0.053.


(np.float64(-0.053), np.float64(0.763))

In [47]:
# タイムスタンプ
# ch05_logging1_timestamp.pyに全体がある
logging.basicConfig(filename="ch05_logs.log",
                    level=logging.DEBUG,
                    filemode="w",  # 以前の内容を上書きする                  
                    format='%(asctime)s %(message)s') # ①

def fit_trendline(year_timestamps, data):
    logging.info("Running fit_trendline function")
    result = linregress(year_timestamps, data)
    slope = round(result.slope, 3)
    r_squared = round(result.rvalue**2, 3)
    logging.info(f"Completed analysis. Slope of the trendline is {slope}.")
    return slope, r_squared

In [49]:
!python3 ch05_logging1_timestamp.py   # 上記の内容を含むファイルを実行
!cat ch05_logs.log    # ログファイルの内容を表示

2025-05-13 22:04:17,587 Running fit_trendline function
2025-05-13 22:04:17,587 Completed analysis. Slope of the trendline is -0.053.


In [14]:
# エラーメッセージをログに記録
# ch05_logging2_errmes.pyに全体がある
def fit_trendline(year_timestamps, data):
    logging.info("Running fit_trendline function")
    try:
        result = linregress(year_timestamps, data)
    except TypeError as e:
        logging.error("Both lists must contain floats or integers.") # ❶
        logging.exception(e) # ❷
    else:
        slope = round(result.slope, 3)
        r_squared = round(result.rvalue**2, 3)
        logging.info(f"Completed analysis. Slope of the trendline is {slope}.")
        return slope, r_squared

In [50]:
!python3 ch05_logging2_errmes.py   # 上記の関数を含むファイルを実行
!cat ch05_logs.log    # ログファイルの内容を表示

2025-05-13 22:08:16,179 Running fit_trendline function
2025-05-13 22:08:16,179 Both lists must contain floats or integers.
2025-05-13 22:08:16,179 ufunc 'maximum' did not contain a loop with signature matching types (dtype('<U4'), dtype('<U4')) -> None
Traceback (most recent call last):
  File "/Users/musha/Desktop/1musha/se4ds/software-engineering-for-data-scientists/src/example/ch05/ch05_logging2_errmes.py", line 11, in fit_trendline
    result = linregress(year_timestamps, data)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/scipy/stats/_stats_py.py", line 10704, in linregress
    if np.amax(x) == np.amin(x) and len(x) > 1:
       ^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/numpy/_core/fromnumeric.py", line 3216, in amax
    return _wrapreduction(a, np.maximum, 'max', axis, None, out,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/numpy/_core/from

## 5.3　デバッグ

In [7]:
# バグのあるコード（ch05_debug1.py）
def weighted_mean(num_list, weights):
    running_total = 0
    for i in range(len(num_list)):
        running_total += num_list[i] * weights[0]  # ←バグ
    return running_total/sum(weights)

In [8]:
weighted_mean([1, 6, 8], [1, 3, 2])

2.5

In [9]:
!python3 ch05_debug1.py

2.5


In [12]:
# print文を挿入（ch05_debug2_print.py）
def weighted_mean(num_list, weights):
    running_total = 0
    for i in range(len(num_list)):
        running_total += num_list[i] * weights[0]
        print(f"The running total at step {i} is {running_total}")
    return running_total / sum(weights)

In [13]:
weighted_mean([1, 6, 8], [1, 3, 2])

The running total at step 0 is 1
The running total at step 1 is 7
The running total at step 2 is 15


2.5

In [14]:
!python3 ch05_debug2_print.py

The running total at step 0 is 1
The running total at step 1 is 7
The running total at step 2 is 15
2.5


In [15]:
# ログに記録（ch05_debug3_log.py）
import logging

logging.basicConfig(filename="ch05_logs.log",
                    level=logging.DEBUG,
                    filemode='w',
                    format='%(asctime)s %(message)s')

def weighted_mean(num_list, weights):
    running_total = 0
    for i in range(len(num_list)):
        running_total += num_list[i] * weights[0]
        logging.debug(f"The running total at step {i} is {running_total}")
    return running_total/sum(weights)


In [16]:
print(weighted_mean([1, 6, 8], [1, 3, 2]))

2.5


In [17]:
!cat ch05_logs.log

2025-05-13 23:37:19,534 The running total at step 0 is 1
2025-05-13 23:37:19,537 The running total at step 1 is 7
2025-05-13 23:37:19,537 The running total at step 2 is 15


In [55]:
numbers = [10, 20, 30, 40, 50]
weights = [0.1, 0.2, 0.3, 0.2, 0.2]

weighted_mean(numbers, weights)

The running total at step 0 is 1.0
The running total at step 1 is 3.0
The running total at step 2 is 6.0
The running total at step 3 is 10.0
The running total at step 4 is 15.0


15.0

In [63]:
# 全体は 
import logging

logging.basicConfig(
    filename="ch05_logs.log", level=logging.DEBUG, format="%(asctime)s %(message)s"
)


def weighted_mean(num_list, weights):
    running_total = 0
    for i in range(len(num_list)):
        running_total += num_list[i] * weights[0]
        logging.debug(f"The running total at step {i} is {running_total}")
    return running_total / len(num_list)

In [64]:
numbers = [10, 20, 30, 40, 50]
weights = [0.1, 0.2, 0.3, 0.2, 0.2]

weighted_mean(numbers, weights)

3.0

!cat ch05_logs.log

### 5.3.2　デバッグ用ツール

別ファイル（[ch05_debugger.ipynb](http://localhost:8888/notebooks/ch05/ch05_debugger.ipynb)）にあります